In [ ]:
import pandas as pd
import numpy as np
import re
import string
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity

# 1. LOAD DATA
df = pd.read_csv(r'C:\Users\Soumya_SRB\Desktop\Netflix-Content-Clustering-Engine\NetflixSimple.csv')

# 2. HANDLE MISSING VALUES
df.fillna('Unknown', inplace=True)

# 3. NLP: CLEAN AND CREATE "BAG OF CONTENT"
def clean_text(text):
    text = str(text).lower()
    text = re.sub(f'[{re.escape(string.punctuation)}]', '', text)
    return text

# Combining metadata fields for semantic understanding
df['text_features'] = (df['director'] + ' ' + df['cast'] + ' ' + 
                      df['listed_in'] + ' ' + df['description']).apply(clean_text)

# 4. VECTORIZATION
tfidf = TfidfVectorizer(stop_words='english', max_features=3000)
tfidf_matrix = tfidf.fit_transform(df['text_features'])

# 5. DIMENSIONALITY REDUCTION (Addressing Scalability)
svd = TruncatedSVD(n_components=50, random_state=42)
matrix_reduced = svd.fit_transform(tfidf_matrix)

# 6. CLUSTERING 
kmeans = KMeans(n_clusters=10, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(matrix_reduced)

# 7. RECOMMENDATION ENGINE 
cosine_sim = cosine_similarity(matrix_reduced)

# 8. INTERACTIVE INTERFACE FUNCTION
def get_recommendations():
    print("\n" + "="*40)
    print(" NETFLIX RECOMMENDATION SYSTEM")
    print("="*40)
    print("Type 'exit' to quit.")
    
    while True:
        title = input("\nEnter a Movie/TV Show Title: ").strip()
        
        if title.lower() == 'exit':
            print("Exiting... Project Complete!")
            break
            
        try:
            # Match title 
            idx = df[df['title'].str.lower() == title.lower()].index[0]
            
            # Compute similarity scores
            sim_scores = list(enumerate(cosine_sim[idx]))
            sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
            
            # Select top 5 (skipping the first one as it is the title itself)
            movie_indices = [i[0] for i in sim_scores[1:6]]
            
            print(f"\nSince you liked '{df.iloc[idx]['title']}', you might enjoy:")
            print("-" * 30)
            print(df[['title', 'type', 'listed_in']].iloc[movie_indices].to_string(index=False))
            print("-" * 30)
            
        except IndexError:
            print(f"Error: '{title}' not found. Please check the spelling and try again.")

# RUN THE INTERFACE
if __name__ == "__main__":
    get_recommendations()


 NETFLIX RECOMMENDATION SYSTEM
Type 'exit' to quit.



Enter a Movie/TV Show Title:  pk



Since you liked 'PK', you might enjoy:
------------------------------
                                           title  type                              listed_in
                                       Aarakshan Movie           Dramas, International Movies
                           English Babu Desi Mem Movie Comedies, Dramas, International Movies
                                         Baazaar Movie           Dramas, International Movies
Shaurya: It Takes Courage to Make Right... Right Movie           Dramas, International Movies
                                 Dil Dhadakne Do Movie Comedies, Dramas, International Movies
------------------------------



Enter a Movie/TV Show Title:  pk



Since you liked 'PK', you might enjoy:
------------------------------
                                           title  type                              listed_in
                                       Aarakshan Movie           Dramas, International Movies
                           English Babu Desi Mem Movie Comedies, Dramas, International Movies
                                         Baazaar Movie           Dramas, International Movies
Shaurya: It Takes Courage to Make Right... Right Movie           Dramas, International Movies
                                 Dil Dhadakne Do Movie Comedies, Dramas, International Movies
------------------------------



Enter a Movie/TV Show Title:  3 idiots



Since you liked '3 Idiots', you might enjoy:
------------------------------
                title  type                              listed_in
      Dil Dhadakne Do Movie Comedies, Dramas, International Movies
English Babu Desi Mem Movie Comedies, Dramas, International Movies
                   PK Movie Comedies, Dramas, International Movies
      Phir Hera Pheri Movie         Comedies, International Movies
 Deewana Main Deewana Movie Comedies, Dramas, International Movies
------------------------------
